# AdaZero-DCE: số vòng lặp thích ứng cho tăng cường ảnh thiếu sáng

**Bài tập lớn môn Mạng nơron và học sâu — Phenikaa**  
Trần Minh Sang — 24100012 · Ngô Quang Thiện — 24102651

Ảnh thiếu sáng → đặc trưng độ sáng → dự đoán số vòng → đường cong Zero-DCE → so sánh fixed-8 / adaptive → kiểm tra khái quát LOLv2-real.

Notebook dùng cấu trúc trình bày của notebook mẫu: giải thích trước, thực thi sau, bảng/hình ngay dưới mỗi bước. **Ba head đã được train trên LOLv1 our485 (485 cặp, 1940 mẫu, 1200 epoch); notebook này chạy lại inference và metric**, không train lại hoặc sửa code gốc. Loss/oracle train được đọc từ artifact full trước đó và ghi rõ nguồn. Tất cả bảng test phía dưới được tính mới trong lần chạy notebook này.

## 1. Môi trường và kiểm tra mô hình
Chạy trên Kaggle GPU T4. Dùng trực tiếp code/trọng số trong output full LOLv1 đã hoàn tất; kiểm tra SHA256 trước khi inference. Không cài thêm thư viện hoặc sửa implementation.

In [ ]:
import os, sys, json, hashlib, time
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
OUT=Path('/kaggle/working/adazero-notebook-results'); OUT.mkdir(exist_ok=True)
FIG=OUT/'figures'; FIG.mkdir(exist_ok=True)
PROJECT=next(Path('/kaggle/input').rglob('adazero/zero_dce_adaptive_n.py')).parent
FULL=next(Path('/kaggle/input').rglob('full/head_seed0.pth')).parent
EVAL_SOURCE=next(Path('/kaggle/input').rglob('lolv2-eval/evaluate_lolv2.py')).parent
sys.path[:0]=[str(PROJECT),str(EVAL_SOURCE)]
from zero_dce_adaptive_n import load_dce, load_head, predict_n, curve_outputs, brightness_features
from experiment import paired_paths, read_image, metrics
expected=json.loads((EVAL_SOURCE/'reports/full/verification.json').read_text())['checkpoint_sha256']
source_hashes=json.loads((EVAL_SOURCE/'reports/full/source_hashes.json').read_text())
for name,h in source_hashes.items():
    assert hashlib.sha256((PROJECT/name).read_bytes()).hexdigest()==h
weights=PROJECT/'AdaZero-DCE_code/snapshots/Epoch99.pth'
head_paths={s:FULL/f'head_seed{s}.pth' for s in [0,1,2]}
model_hashes={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [weights,*head_paths.values()]}
assert all(model_hashes[n]==h for n,h in expected.items())
torch.set_num_threads(2); torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark=False
net=load_dce(weights,'cuda')
heads={s:load_head(p) for s,p in head_paths.items()}
for h,_,_ in heads.values(): h.requires_grad_(False)
plt.rcParams.update({'figure.dpi':110,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
display(pd.DataFrame({'Thành phần':['GPU','PyTorch','Head','Backbone','Chuẩn hóa'],
                     'Giá trị':[torch.cuda.get_device_name(0),torch.__version__,'3 seed 0/1/2, đóng băng','Epoch99, đóng băng','mu/sd LOLv1 giữ nguyên']}))

## 2. Chuẩn bị dữ liệu và giao thức
LOLv1: 15 cặp eval15, bốn điều kiện (low gốc; GT×0.3/0.45/0.6), tổng 60 trường hợp/seed. Không dùng GT để chọn n. LOLv2-real: chỉ tải 100 cặp Test, không tải ảnh Train. Các ảnh giữ nguyên RGB 600×400, không crop/resize hay cân bằng sáng theo GT.

Ảnh GT nhân hệ số là **giả lập sRGB**, không mô phỏng đầy đủ quá trình chụp ảnh thiếu sáng và nhiễu cảm biến.

In [ ]:
LOL=next(Path('/kaggle/input').rglob('our485')).parent
pairs=paired_paths(LOL,'eval15',15)
train_manifest=json.loads((FULL/'manifest.json').read_text())
assert train_manifest['train_pairs']==485 and train_manifest['epochs']==1200
cache=np.load(FULL/'train_cache.npz')
assert cache['features'].shape==(1940,10)
display(pd.DataFrame([
 {'Tập':'LOLv1 our485','Cặp gốc':485,'Input':1940,'Vai trò':'Huấn luyện đã hoàn tất'},
 {'Tập':'LOLv1 eval15','Cặp gốc':15,'Input':60,'Vai trò':'Đo mới, 4 điều kiện'},
 {'Tập':'LOLv2-real Test','Cặp gốc':100,'Input':100,'Vai trò':'Đo mới, ảnh thật'}]))
low0,gt0=map(read_image,pairs[0])
fig,axes=plt.subplots(1,5,figsize=(15,3))
for ax,img,title in zip(axes,[low0,gt0*.3,gt0*.45,gt0*.6,gt0],['Low goc','GT x 0.3','GT x 0.45','GT x 0.6','GT']):
    ax.imshow(img);ax.set_title(title);ax.axis('off')
fig.tight_layout();fig.savefig(FIG/'data_conditions.png',dpi=220,bbox_inches='tight');plt.show()

## 3. DCE-Net và head hồi quy
DCE-Net sinh tám bản đồ hệ số một lần cho mỗi ảnh. Với quy ước dấu của code gốc:
$$x_t=x_{t-1}+r_t(x_{t-1}^2-x_{t-1}).$$
Head 10→128→64→1 dùng mean/std của trung bình RGB và histogram 8 bin. Mean < 0.10 giữ n=8; còn lại round/clamp về [1,8]. Vì DCE-Net vẫn chạy đầy đủ, số vòng giảm không đồng nghĩa tốc độ toàn mạng tăng cùng tỷ lệ.

In [ ]:
h,mu,sd=heads[0]
display(h)
display(pd.DataFrame({'Đặc trưng':['mean','std']+[f'hist_{i}' for i in range(8)],
                     'Input đầu tiên':brightness_features(low0),'mu train':mu,'sd train':sd}))
print('Số tham số head:',sum(p.numel() for p in h.parameters()))
print('Mean ảnh đầu:',float(low0.mean()),'; n seed 0:',predict_n(low0,*heads[0]))

## 4. Huấn luyện đã hoàn tất và phân bố oracle
Nguồn: output full LOLv1 trước đó. Cấu hình cố định: MLP 10-128-64-1, dropout 0.1, SmoothL1, Adam 5e-3, CosineAnnealing, 1200 epoch, full batch. Oracle chỉ được tính từ our485. Các biểu đồ dưới đây **không phải một lần train mới**; chúng được dựng lại từ loss/cache đã lưu.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,3.8))
for s in [0,1,2]:
    losses=json.loads((FULL/f'loss_seed{s}.json').read_text());assert len(losses)==1200
    axes[0].plot(np.arange(1,1201),losses,label=f'Seed {s}',lw=1)
axes[0].set(xlabel='Epoch',ylabel='SmoothL1',title='Lich su huan luyen LOLv1');axes[0].legend()
counts=np.bincount(cache['targets'].astype(int),minlength=9)[1:9]
axes[1].bar(range(1,9),counts,color='#356d92');axes[1].set(xticks=range(1,9),xlabel='Oracle n',ylabel='So mau',title='1940 nhan train')
fig.tight_layout();fig.savefig(FIG/'training.png',dpi=240);plt.show()
display(pd.DataFrame({'oracle_n':range(1,9),'Số mẫu':counts}))

## 5. Đánh giá mới trên LOLv1 eval15
Fixed-8/adaptive dùng cùng hàm PSNR/SSIM. PSNR RGB data_range=1; SSIM RGB, cửa sổ đều 7×7, sample covariance. Trung bình metric từng ảnh; báo toàn bộ 3 seed, không chọn seed tốt nhất. Không so số SSIM tuyệt đối với nguồn có giao thức khác.

In [ ]:
rows=[]
conditions=[('LOL goc',1.),('GT x 0.3',.3),('GT x 0.45',.45),('GT x 0.6',.6)]
start=time.time()
with torch.inference_mode():
    for lp,gp in pairs:
        low,gt=read_image(lp),read_image(gp)
        for condition,factor in conditions:
            img=low if factor==1 else gt*factor
            choices={s:predict_n(img,*head) for s,head in heads.items()}
            needed=set(choices.values())|{8}
            outputs={n:x for n,x in enumerate(curve_outputs(img,net),1) if n in needed}
            bp,bs=metrics(outputs[8],gt)
            for seed,n in choices.items():
                ap,ass=metrics(outputs[n],gt)
                rows.append(dict(file=lp.name,condition=condition,factor=factor,seed=seed,n=n,
                                 input_mean=float(img.mean()),fixed_psnr=bp,adaptive_psnr=ap,delta_psnr=ap-bp,
                                 fixed_ssim=bs,adaptive_ssim=ass,delta_ssim=ass-bs))
            if lp==pairs[0][0]:
                fig,axes=plt.subplots(1,4,figsize=(12,2.8))
                for ax,x,title in zip(axes,[img,outputs[8],outputs[choices[0]],gt],['Input','Fixed-8',f'Adaptive n={choices[0]}','GT']):
                    ax.imshow(x.clip(0,1));ax.set_title(title);ax.axis('off')
                fig.suptitle(condition);fig.tight_layout()
                fig.savefig(FIG/f'comparison_{factor}.png',dpi=220,bbox_inches='tight');plt.show()
v1=pd.DataFrame(rows);assert len(v1)==180
v1.to_csv(OUT/'lolv1_per_image.csv',index=False)
keys=['fixed_psnr','adaptive_psnr','delta_psnr','fixed_ssim','adaptive_ssim','delta_ssim','n']
v1_seed=v1.groupby(['condition','factor','seed'],sort=False)[keys].mean().reset_index()
v1_seed.to_csv(OUT/'lolv1_summary.csv',index=False)
display(v1_seed.round(6))
print('Thời gian đánh giá LOLv1:',round(time.time()-start,2),'giây')

In [ ]:
v1_mean=v1_seed.groupby(['condition','factor'],sort=False)[keys].mean().reset_index()
v1_std=v1_seed.groupby(['condition','factor'],sort=False)[keys].std(ddof=0).reset_index()
v1_mean.to_csv(OUT/'lolv1_mean.csv',index=False);v1_std.to_csv(OUT/'lolv1_std.csv',index=False)
display(Markdown('### Trung bình 3 seed và độ lệch chuẩn'))
display(v1_mean.round(6));display(v1_std[['condition','adaptive_psnr','adaptive_ssim']].round(6))
fig,axes=plt.subplots(1,2,figsize=(10,3.8))
for ax,key,title in zip(axes,['delta_psnr','delta_ssim'],['Delta PSNR (dB)','Delta SSIM']):
    ax.bar(v1_mean['condition'],v1_mean[key],yerr=v1_std[key],capsize=4,color=['#818b98','#bf7850','#387da0','#387da0'])
    ax.axhline(0,color='black',lw=.8);ax.set_title(title);ax.tick_params(axis='x',rotation=15)
fig.tight_layout();fig.savefig(FIG/'lolv1_deltas.png',dpi=240);plt.show()

## 6. Kiểm tra khái quát trên LOLv2-real
Tải đúng 100 Low + 100 Normal từ thư mục Test của mirror Kaggle; kiểm tra file, kích thước và checksum. Dùng nguyên cả ba head LOLv1, không fit normalization, không tạo oracle và không đọc LOLv2 Train. Ghép theo ID low00690 ↔ normal00690.

In [ ]:
import io,csv,re,urllib.request
from urllib.parse import quote
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
source_rows=list(csv.DictReader((EVAL_SOURCE/'reports/lolv2-real/source_files.csv').open()))
assert len(source_rows)==200
TEST=Path('/kaggle/working/lolv2-test')
def download_test(row):
    name=row['name']
    assert re.fullmatch(r'LOL-v2/Real_captured/Test/(Low/low|Normal/normal)\d{5}\.png',name)
    url='https://www.kaggle.com/api/v1/datasets/download/tanhyml/lol-v2-dataset/'+quote(name,safe='')
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url,timeout=90) as response:data=response.read()
            assert len(data)==int(row['size'])
            with Image.open(io.BytesIO(data)) as im:im.verify()
            dest=TEST/Path(name).relative_to('LOL-v2/Real_captured/Test')
            dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
            return dict(source=name,bytes=len(data),sha256=hashlib.sha256(data).hexdigest())
        except Exception:
            if attempt==2:raise
            time.sleep(1)
with ThreadPoolExecutor(max_workers=4) as pool: downloads=list(pool.map(download_test,source_rows))
assert len(list(TEST.rglob('*.png')))==200
(OUT/'downloads.json').write_text(json.dumps(downloads,indent=2))
print('Đã tải 200 ảnh test, tổng byte:',sum(x['bytes'] for x in downloads))

In [ ]:
from types import SimpleNamespace
from evaluate_lolv2 import run as evaluate_lolv2
V2=OUT/'lolv2'
evaluate_lolv2(SimpleNamespace(test=TEST,heads=FULL,weights=weights,
                              verification=EVAL_SOURCE/'reports/full/verification.json',output=V2))
v2=pd.read_csv(V2/'per_image.csv')
v2_seed=v2.groupby('seed')[keys].mean().reset_index()
v2_seed.to_csv(OUT/'lolv2_summary.csv',index=False)
display(v2_seed.round(6))
display(pd.DataFrame({'Chỉ số':['PSNR adaptive','Delta PSNR','SSIM adaptive','Delta SSIM'],
                     'Mean':[v2_seed[k].mean() for k in ['adaptive_psnr','delta_psnr','adaptive_ssim','delta_ssim']],
                     'Std seed':[v2_seed[k].std(ddof=0) for k in ['adaptive_psnr','delta_psnr','adaptive_ssim','delta_ssim']]}).round(6))

## 7. Phân tích kết quả và giới hạn
Phân biệt ba tình huống: ảnh được guard n=8; ảnh được head chọn ít vòng hơn; dữ liệu giả lập và ảnh thật. Std qua 3 seed chỉ đo độ nhạy theo khởi tạo, không phải khoảng tin cậy theo ảnh. Không điều chỉnh threshold hoặc chọn mô hình bằng tập test.

In [ ]:
guard=v2[v2.seed==0].copy()
fig,axes=plt.subplots(1,2,figsize=(10,3.8))
axes[0].hist(guard.input_mean,bins=20,color='#387da0',edgecolor='white')
axes[0].axvline(.10,color='#b35136',ls='--',label='Guard 0.10');axes[0].legend()
axes[0].set(xlabel='Mean RGB',ylabel='So anh',title='Do sang LOLv2-real')
for s in [0,1,2]:
    selected=v2[v2.seed==s]
    axes[1].scatter(selected.input_mean,selected.delta_psnr,label=f'Seed {s}',s=22,alpha=.65)
axes[1].axhline(0,color='black',lw=.8);axes[1].set(xlabel='Mean RGB',ylabel='Delta PSNR (dB)',title='Anh huong cua adaptive');axes[1].legend()
fig.tight_layout();fig.savefig(FIG/'lolv2_analysis.png',dpi=240);plt.show()
counts=[]
for s in [0,1,2]:
    z=v2[v2.seed==s]
    counts.append({'seed':s,'guard':int((z.input_mean<.1).sum()),'tăng PSNR':int((z.delta_psnr>0).sum()),
                   'bằng':int((z.delta_psnr==0).sum()),'giảm':int((z.delta_psnr<0).sum()),'n TB':z.n.mean()})
display(pd.DataFrame(counts))
display(Markdown(f'**LOLv2-real:** ΔPSNR trung bình = {v2_seed.delta_psnr.mean():+.4f} dB; ΔSSIM = {v2_seed.delta_ssim.mean():+.6f}. '
                 f'Có {(guard.input_mean<.1).sum()}/100 ảnh giữ n=8 theo guard. Kết luận cần xét đồng thời PSNR, SSIM và phạm vi dữ liệu.'))

## 8. Xuất kết quả và xác minh
Lưu bảng CSV, hình biểu đồ và manifest. So sánh lần tái đánh giá này với artifact cũ là bước kiểm tra tái lập; số liệu báo cáo lấy từ CSV mới. Không thay hoặc ghi đè checkpoint.

Bản notebook có outputs được thực thi bởi `nbclient` ngay trên Kaggle và xuất thành file riêng để tải về. Outputs là kết quả thực thi thực tế, không điền tay từ log.

In [ ]:
previous_v1=pd.read_csv(FULL/'summary.csv')
for _,row in v1_seed.iterrows():
    old=previous_v1[(previous_v1.factor==row.factor)&(previous_v1.seed==row.seed)].iloc[0]
    np.testing.assert_allclose(row[keys].to_numpy(float),old[keys].to_numpy(float),atol=1e-6,rtol=0)
previous_v2=json.loads(next(Path('/kaggle/input').rglob('lolv2-results/results.json')).read_text())['summary']
for old in previous_v2:
    row=v2_seed[v2_seed.seed==old['seed']].iloc[0]
    np.testing.assert_allclose(row[keys].to_numpy(float),[old[k] for k in keys],atol=1e-6,rtol=0)
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==model_hashes[p.name] for p in [weights,*head_paths.values()])
manifest=dict(seeds=[0,1,2],lolv1_records=len(v1),lolv2_records=len(v2),model_hashes=model_hashes,
              source_hashes=source_hashes,training='Historical full LOLv1 artifact, not rerun',
              evaluation='New inference and metrics in this notebook',reproduction_atol=1e-6,
              gpu=torch.cuda.get_device_name(0),torch=torch.__version__)
(OUT/'notebook_manifest.json').write_text(json.dumps(manifest,indent=2))
import shutil
shutil.rmtree(TEST)
print('PASS: 180 dòng LOLv1 + 300 dòng LOLv2; khớp kết quả trước trong 1e-6; checkpoint không đổi.')
print('Đã xuất CSV, hình và manifest:',OUT)